# Sampling and Aliasing — Interactive Demo

**Topic 4: Sampling and Aliasing** · Digital Signal Processing for Sound and Music

Inspired by the DSPFirst **Continuous–Discrete Sampling Demo (`con2dis`)** and aligned with the course's Topic 4 summary.

**Learning goals**

1. Connect continuous time and discrete time: $x[n]=x(nT_s)$, with $T_s=1/f_s$.
2. Change the input frequency and sampling rate, and identify **aliases** in both the waveform and spectrum.
3. Understand **spectral folding** around the Nyquist frequency $f_s/2$.
4. Compare sampling a **single sinusoid** with sampling a **three-tone signal**.
5. Distinguish reconstruction methods and understand why an **anti-aliasing filter** must be applied **before** sampling.

Run the cells in order. In Jupyter Notebook / JupyterLab, the interactive controls use `ipywidgets` (install with `%pip install ipywidgets` if needed). No audio files or online services are required.

**References:** McClellan, Schafer & Yoder, *Signal Processing First*, Topic 4 summary; [DSPFirst `con2dis` demo](https://dspfirst.gatech.edu/matlab/#con2dis).

## 1 · Sampling, aliases, and normalized frequency

For a continuous-time cosine

$$x(t)=A\cos(2\pi f_0t+\phi),$$

uniform sampling at $f_s$ gives

$$x[n]=x(nT_s)=A\cos\!\left(2\pi\frac{f_0}{f_s}n+\phi\right),\qquad T_s=\frac{1}{f_s}.$$

Its **normalized radian frequency** is $\hat\omega_0=2\pi f_0/f_s$ (radians/sample). Because discrete-time frequency is periodic with period $2\pi$, analogue frequencies related by $f_a=\pm f_0+kf_s$ can give identical samples (with the corresponding phase change for a folded cosine).

The representative *signed* frequency in the principal Nyquist interval $[-f_s/2,f_s/2)$ is

$$f_{\mathrm{signed}}=\left((f_0+f_s/2)\bmod f_s\right)-f_s/2,$$

and the **apparent nonnegative frequency** is $f_{\mathrm{alias}}=|f_{\mathrm{signed}}|\le f_s/2$.

**Important:** Folding changes the phase sign of a real cosine when its signed alias is negative. At DC or exactly Nyquist, the sampled phase cannot be fully recovered. An ideal bandlimited reconstruction selects the baseband alias—it cannot determine which analogue alias produced the samples.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown

try:
    import ipywidgets as widgets
    WIDGETS_OK = True
except ImportError:
    widgets = None
    WIDGETS_OK = False

plt.rcParams.update({'figure.dpi': 115, 'axes.grid': True, 'grid.alpha': 0.25})
print('Interactive widgets available:', WIDGETS_OK)


Interactive widgets available: True


## 2 · Signal construction and folding calculations

The demonstrations use analytical continuous-time signals (not signals synthesized on a high-rate grid and then incorrectly downsampled). The **three-tone signal** contains components at $f_0$, $2.7f_0$ and $4.3f_0$, each with its own amplitude and phase. This makes it easy to see how separate frequencies fold to different places in the Nyquist interval.

The optional *ideal anti-alias filter* removes components at or above $f_s/2$ **before sampling**. It is a pedagogical frequency-domain model of prefiltering, not a practical FIR/IIR filter.


In [2]:
def make_components(kind='Single sinusoid', f0=100., phase_deg=0.):
    """Return (frequency in Hz, amplitude, phase in rad) for each analogue cosine."""
    phi = np.deg2rad(phase_deg)
    if kind == 'Single sinusoid':
        return [(float(f0), 1.0, phi)]
    if kind == 'Three-tone signal':
        return [(float(f0), 1.00, phi),
                (2.7 * float(f0), 0.65, phi + 0.60),
                (4.3 * float(f0), 0.45, phi - 0.80)]
    if kind == 'Harmonic-rich signal':
        return [(k * float(f0), 1.0 / k, phi - 0.20*k)
                for k in range(1, 7)]
    raise ValueError(f'Unknown signal: {kind}')


def signal_at(t, components):
    t = np.asarray(t)
    y = np.zeros_like(t, dtype=float)
    for freq, amp, phase in components:
        y += amp * np.cos(2 * np.pi * freq * t + phase)
    return y


def signed_alias(f, fs):
    """Wrap f to [-fs/2, fs/2); retains sign for phase-correct reconstruction."""
    return (float(f) + fs / 2) % fs - fs / 2


def baseband_signal(t, components, fs):
    """Ideal low-pass representative of the samples of a sum of cosines."""
    t = np.asarray(t)
    y = np.zeros_like(t, dtype=float)
    for freq, amp, phase in components:
        y += amp * np.cos(2 * np.pi * signed_alias(freq, fs) * t + phase)
    return y


def ideal_pre_filter(components, fs):
    return [(f, a, ph) for f, a, ph in components if f < fs/2 - 1e-10]


def folded_spectral_lines(components, fs):
    """One-sided line amplitudes, with correct phasor addition if aliases coincide."""
    grouped = {}
    for f, amp, phase in components:
        signed = signed_alias(f, fs)
        af = round(abs(signed), 9)
        if np.isclose(af, 0.) or np.isclose(af, fs/2):
            coeff = complex(amp * np.cos(phase), 0.0)
        else:
            coeff = amp * np.exp(1j * (phase if signed >= 0 else -phase))
        grouped[af] = grouped.get(af, 0j) + coeff
    return sorted([(f, abs(coeff)) for f, coeff in grouped.items()
                   if abs(coeff) > 1e-10])


def original_spectral_lines(components):
    grouped = {}
    for f, amp, phase in components:
        grouped[f] = grouped.get(f, 0j) + amp * np.exp(1j * phase)
    return sorted((f, abs(c)) for f, c in grouped.items() if abs(c) > 1e-10)


def finite_sinc_reconstruction(t, components, fs, extension_s=0.22):
    """Finite approximation to ideal sinc reconstruction, with samples beyond plot ends."""
    n_min = int(np.floor(-extension_s * fs))
    n_max = int(np.ceil((float(np.max(t)) + extension_s) * fs))
    n = np.arange(n_min, n_max + 1)
    yn = signal_at(n / fs, components)
    return np.sinc(fs * t[:, None] - n[None, :]) @ yn


def reconstruction_values(t, components, fs, mode):
    if mode == 'Ideal baseband':
        return baseband_signal(t, components, fs)
    if mode == 'Finite sinc':
        return finite_sinc_reconstruction(t, components, fs)
    n = np.arange(int(np.ceil(float(t[-1]) * fs)) + 2)
    ts = n / fs
    samples = signal_at(ts, components)
    if mode == 'Linear':
        return np.interp(t, ts, samples)
    if mode == 'Sample-and-hold':
        return samples[np.minimum(np.floor(fs*t).astype(int), len(samples)-1)]
    raise ValueError(f'Unknown reconstruction: {mode}')


## 3 · Interactive continuous-to-discrete demonstration

Experiment with the **sampling rate** ($f_s$), **frequency** ($f_0$), **initial phase**, reconstruction method, and display duration. Try sampling at $f_s=250$, $125$, $100$, and $80$ Hz for a 100-Hz sine.

The plot shows:

1. The original continuous-time signal, the samples, and a reconstruction from the samples.
2. The original one-sided **line spectrum** (theoretical sinusoid amplitudes).
3. The spectrum as seen **inside the Nyquist band after sampling**. When aliases coincide, their phases are combined.
4. The **folding curve**, with the signal's frequencies marked on it.

For the spectral plots, **Hz** is the default. You can also display normalized $\hat\omega = 2\pi f/f_s$ (radians/sample). These are theoretical line spectra, not windowed FFT estimates.


In [3]:
def _stem_lines(ax, lines, label, marker_color='tab:blue'):
    if lines:
        freqs, mags = zip(*lines)
        ax.vlines(freqs, 0, mags, color=marker_color, linewidth=2.0, label=label)
        ax.plot(freqs, mags, 'o', color=marker_color, markersize=5)
    ax.axhline(0, color='0.3', linewidth=.7)


def plot_sampling_demo(kind='Single sinusoid', f0=100., fs=125.,
                       phase_deg=0., duration_ms=85.,
                       reconstruction='Ideal baseband',
                       anti_alias=False, freq_axis='Hz'):
    fs = float(fs); f0 = float(f0)
    original = make_components(kind, f0, phase_deg)
    active = ideal_pre_filter(original, fs) if anti_alias else original
    T = float(duration_ms) / 1000
    t = np.linspace(0, T, 1400)
    n = np.arange(int(np.ceil(T*fs)) + 1)
    tn = n / fs
    tn = tn[tn <= T + 1e-12]
    xn = signal_at(tn, active)
    original_t = signal_at(t, original)
    filtered_t = signal_at(t, active)
    reconstructed = reconstruction_values(t, active, fs, reconstruction)

    x_axis = (lambda f: np.asarray(f)*2*np.pi/fs) if freq_axis != 'Hz' else np.asarray
    unit = 'Normalized frequency (rad/sample)' if freq_axis != 'Hz' else 'Frequency (Hz)'
    nyquist = np.pi if freq_axis != 'Hz' else fs/2
    frequencies = [f for f, _, _ in original]
    upper_hz = max(max(frequencies)*1.12, fs*.62)
    upper = float(x_axis(upper_hz))

    fig = plt.figure(figsize=(11.0, 10.0), constrained_layout=True)
    grid = fig.add_gridspec(4, 1, height_ratios=(1.50, .83, .83, 1.10))
    a0, a1, a2, a3 = (fig.add_subplot(grid[i, 0]) for i in range(4))

    a0.plot(t*1000, original_t, linewidth=1.35, label='Original continuous signal')
    if anti_alias:
        a0.plot(t*1000, filtered_t, '--', linewidth=1.35, label='After ideal anti-alias filter')
    a0.plot(t*1000, reconstructed, '--', linewidth=1.7, label=f'{reconstruction} reconstruction')
    a0.plot(tn*1000, xn, 'o', ms=5, label='Samples x[n]', zorder=5)
    a0.set_ylabel('Amplitude')
    a0.set_xlabel('Time (ms)')
    a0.set_title(f'{kind} — fₛ = {fs:g} samples/s, Nyquist = {fs/2:g} Hz')
    a0.legend(loc='upper right', fontsize=8, ncol=2)

    original_lines = original_spectral_lines(original)
    filtered_lines = original_spectral_lines(active)
    _stem_lines(a1, [(float(x_axis(f)), a) for f, a in original_lines],
                'Original frequencies', 'tab:blue')
    if anti_alias:
        _stem_lines(a1, [(float(x_axis(f)), a) for f, a in filtered_lines],
                    'Passed by anti-alias filter', 'tab:green')
    a1.axvline(nyquist, color='0.3', ls=':', label='Nyquist')
    a1.set_xlim(0, upper)
    a1.set_ylabel('Amplitude')
    a1.set_title('Before sampling: analogue line spectrum')
    a1.legend(loc='upper right', fontsize=8)

    aliases = folded_spectral_lines(active, fs)
    _stem_lines(a2, [(float(x_axis(f)), a) for f, a in aliases],
                'Aliased baseband frequencies', 'tab:orange')
    a2.axvspan(0, nyquist, color='tab:orange', alpha=.09)
    a2.axvline(nyquist, color='0.3', ls=':', label='Nyquist')
    a2.set_xlim(0, upper)
    a2.set_xlabel(unit)
    a2.set_ylabel('Amplitude')
    a2.set_title('After sampling: apparent frequencies in the baseband')
    a2.legend(loc='upper right', fontsize=8)

    fcurve = np.linspace(0, upper_hz, 900)
    foldcurve = np.abs((fcurve + fs/2) % fs - fs/2)
    a3.plot(x_axis(fcurve), x_axis(foldcurve), linewidth=1.65, label='Folding relation')
    a3.plot(x_axis(fcurve[fcurve <= fs/2]), x_axis(fcurve[fcurve <= fs/2]),
            ls=':', alpha=.65, label='No folding: f_alias = f')
    for i, (f, amp, _) in enumerate(original):
        af = abs(signed_alias(f, fs))
        x, y = float(x_axis(f)), float(x_axis(af))
        a3.scatter(x, y, s=38, zorder=4)
        a3.annotate(f'{f:g} → {af:g} Hz', (x, y), xytext=(6, 8 + 6*(i % 2)),
                    textcoords='offset points', fontsize=8)
    a3.axhline(nyquist, color='0.4', ls='--', lw=.9)
    a3.axvline(nyquist, color='0.4', ls='--', lw=.9)
    a3.set_xlim(0, upper)
    a3.set_ylim(0, 1.25 * nyquist)
    a3.set_xlabel(f'Input {unit.lower()}')
    a3.set_ylabel(f'Apparent {unit.lower()}')
    a3.set_title('Spectral folding: each component is reflected into [0, fₛ/2]')

    plt.show()

    if not active:
        print(f'Ideal anti-alias prefilter: no components below {fs/2:g} Hz; output is zero.')
        return
    print('Component frequencies before sampling → signed alias → apparent frequency:')
    for f, amp, ph in active:
        s = signed_alias(f, fs)
        flag = 'FOLDED' if f > fs/2 else ('NYQUIST' if np.isclose(f, fs/2) else 'baseband')
        print(f'  {f:7.1f} Hz → {s:7.1f} Hz → {abs(s):6.1f} Hz  [{flag}]')
    check = np.max(np.abs(xn - baseband_signal(tn, active, fs)))
    print(f'Agreement of sampled values with their baseband alias: max error = {check:.2e}')


In [4]:
def sampling_controls(kind, fs_start, f0_start=100):
    """Create a self-contained lecture demo; works without widgets as a static figure."""
    if not WIDGETS_OK:
        print('Install ipywidgets to enable controls: %pip install ipywidgets')
        return plot_sampling_demo(kind=kind, fs=fs_start, f0=f0_start)

    style = {'description_width': 'initial'}
    fs_ctl = widgets.IntSlider(value=fs_start, min=60, max=1000, step=5,
                               description='Sampling rate fₛ (Hz)', style=style,
                               continuous_update=False)
    f0_ctl = widgets.IntSlider(value=f0_start, min=40, max=350, step=5,
                               description='Reference frequency f₀ (Hz)', style=style,
                               continuous_update=False)
    phase_ctl = widgets.IntSlider(value=0, min=-180, max=180, step=15,
                                  description='Phase (degrees)', style=style,
                                  continuous_update=False)
    duration_ctl = widgets.IntSlider(value=85, min=40, max=160, step=5,
                                     description='Display duration (ms)', style=style,
                                     continuous_update=False)
    method_ctl = widgets.Dropdown(options=['Ideal baseband', 'Finite sinc',
                                           'Linear', 'Sample-and-hold'],
                                  value='Ideal baseband', description='Reconstruction', style=style)
    filter_ctl = widgets.Checkbox(value=False, description='Ideal anti-alias filter (before sampling)',
                                  indent=False)
    axis_ctl = widgets.Dropdown(options=['Hz', 'Normalized rad/sample'],
                                value='Hz', description='Frequency axis', style=style)
    controls = dict(f0=f0_ctl, fs=fs_ctl, phase_deg=phase_ctl,
                    duration_ms=duration_ctl, reconstruction=method_ctl,
                    anti_alias=filter_ctl, freq_axis=axis_ctl)
    output = widgets.interactive_output(lambda **kwargs: plot_sampling_demo(kind=kind, **kwargs), controls)
    left = widgets.VBox([fs_ctl, f0_ctl, phase_ctl, duration_ctl])
    right = widgets.VBox([method_ctl, filter_ctl, axis_ctl])
    display(widgets.HBox([left, right]), output)
    return controls


### 3.1 · A single sinusoid: samples can fit a different frequency

Start with **100 Hz sampled at 125 Hz**. The samples of a 100-Hz cosine are indistinguishable from those of a **25-Hz cosine** with appropriate phase: $100 = 125-25$.

Set $f_s$ to 250 Hz (no alias), 125 Hz (25-Hz folded alias), 100 Hz (0-Hz/DC alias at phase 0), then 80 Hz (20-Hz alias). Experiment with phase $\phi=60^\circ$ to see why the sign of the folded frequency matters.


In [5]:
single_controls = sampling_controls('Single sinusoid', fs_start=125, f0_start=100)

Output()

### 3.2 · A complex signal: three different frequencies fold differently

Use the composite signal

$$x(t)=\cos(2\pi f_0t+\phi)+0.65\cos(2\pi(2.7f_0)t+\phi+0.6)+0.45\cos(2\pi(4.3f_0)t+\phi-0.8).$$

With $f_0=100$ Hz and $f_s=300$ Hz, its 100-, 270-, and 430-Hz components appear at **100, 30, and 130 Hz** respectively. Raise $f_s$ to 900 Hz to avoid aliasing of all three. Toggle the ideal anti-alias filter to see what is *removed* rather than folded.


In [6]:
complex_controls = sampling_controls('Three-tone signal', fs_start=300, f0_start=100)

Output()

## 4 · Spectral replicas: why different frequencies produce the same samples

Sampling a sinusoid creates spectral copies separated by $f_s$. In discrete time, the two-sided frequency axis is periodic: equivalent components occur at $\pm f_0+kf_s$, with $k\in\mathbb{Z}$. Only the central interval $[-f_s/2,f_s/2]$ is the principal Nyquist band.

The plot below is **schematic line-spectrum placement** for a cosine, not an FFT of a finite sequence. Observe how changing $f_s$ moves the copies and which pair falls inside the shaded interval.


In [7]:
def plot_spectral_replicas(f0=100., fs=125.):
    f0 = float(f0); fs=float(fs)
    base = [q for k in range(-4, 5) for q in (k*fs-f0, k*fs+f0)]
    lim = 2.6*fs
    peaks = [f for f in base if -lim <= f <= lim]
    fig, ax = plt.subplots(figsize=(10.5, 3.8))
    ax.axvspan(-fs/2, fs/2, color='tab:blue', alpha=.12,
               label='Principal Nyquist band')
    ax.vlines(peaks, 0, 1, color='0.5', linewidth=1.3)
    ax.plot(peaks, np.ones(len(peaks)), '.', color='0.35')
    folded = abs(signed_alias(f0, fs))
    ax.axvline(-fs/2, color='tab:blue', ls='--', lw=1)
    ax.axvline(fs/2, color='tab:blue', ls='--', lw=1)
    ax.vlines([-folded, folded], 0, 1.15, color='tab:red', lw=2.5,
              label=f'Components in Nyquist band: ±{folded:g} Hz')
    ax.set(xlim=(-lim,lim), ylim=(0,1.32), xlabel='Frequency (Hz)',
           ylabel='Schematic magnitude', title=f'Sampled sinusoid: f₀={f0:g} Hz, fₛ={fs:g} Hz')
    ax.legend(loc='upper right')
    plt.tight_layout(); plt.show()

if WIDGETS_OK:
    widgets.interact(plot_spectral_replicas,
                     f0=widgets.IntSlider(value=100, min=40,max=350,step=5,continuous_update=False),
                     fs=widgets.IntSlider(value=125,min=60,max=1000,step=5,continuous_update=False))
else:
    plot_spectral_replicas()


interactive(children=(IntSlider(value=100, continuous_update=False, description='f0', max=350, min=40, step=5)…

## 5 · Listen: original sound versus its baseband alias

Both audio players use a **fixed output sampling rate of 16 kHz**. The first synthesizes the original analogue-frequency components; the second synthesizes the ideal **baseband representative** that matches the chosen low-rate samples. This isolates the *effect of aliasing* from the unrelated pitch change that would occur if we played the samples with the wrong declared sample rate.

Use the buttons to hear a 100-Hz sinusoid or the three-tone signal. Change $f_s$, or enable the ideal anti-alias filter, then click **Play comparison** again.


In [8]:
AUDIO_FS = 16000

def make_listening_pair(kind, f0=100, fs=125, phase_deg=0, anti_alias=False, duration_s=1.25):
    comps = make_components(kind, f0, phase_deg)
    kept = ideal_pre_filter(comps, fs) if anti_alias else comps
    t = np.arange(int(duration_s*AUDIO_FS)) / AUDIO_FS
    original = signal_at(t, comps)
    reconstructed = baseband_signal(t, kept, fs)
    # Same gain for a fair level comparison and no clipping.
    peak = max(np.max(np.abs(original)), np.max(np.abs(reconstructed)), 1e-9)
    return 0.90*original/peak, 0.90*reconstructed/peak

if WIDGETS_OK:
    a_kind = widgets.Dropdown(options=['Single sinusoid','Three-tone signal','Harmonic-rich signal'],
                              description='Signal')
    a_fs = widgets.IntSlider(value=125,min=60,max=1000,step=5,description='Sampling rate (Hz)',continuous_update=False)
    a_f0 = widgets.IntSlider(value=100,min=40,max=350,step=5,description='f₀ (Hz)',continuous_update=False)
    a_filter = widgets.Checkbox(value=False,description='Ideal anti-alias filter')
    play_button = widgets.Button(description='Play comparison', button_style='info')
    audio_out = widgets.Output()

    def on_play(_):
        original, alias = make_listening_pair(a_kind.value, a_f0.value, a_fs.value,
                                             anti_alias=a_filter.value)
        with audio_out:
            audio_out.clear_output(wait=True)
            display(Markdown('**Original signal:**'))
            display(Audio(original, rate=AUDIO_FS, normalize=False))
            display(Markdown('**Ideal baseband reconstruction from the samples:**'))
            display(Audio(alias, rate=AUDIO_FS, normalize=False))
    play_button.on_click(on_play)
    display(widgets.VBox([a_kind,a_fs,a_f0,a_filter,play_button,audio_out]))
else:
    print('Interactive audio comparison needs ipywidgets. Running a static pair:')
    org, alias = make_listening_pair('Single sinusoid', 100, 125)
    display(Audio(org, rate=AUDIO_FS, normalize=False))
    display(Audio(alias, rate=AUDIO_FS, normalize=False))


## 6 · Exercises and takeaways

1. **No aliasing:** Set the single cosine to $f_0=100$ Hz and $f_s=250$ Hz. What is the Nyquist frequency? How do the analogue and reconstructed curves compare?
2. **Folding:** Set $f_s=125$ Hz. Verify graphically and algebraically that $100$ Hz aliases to $25$ Hz. Change the phase to $60^\circ$: does the same positive-phase 25-Hz cosine go through the sample values?
3. **At the sampling rate:** Set $f_s=f_0=100$ Hz and $\phi=0$. Why are all samples equal to 1? What if $\phi=90^\circ$?
4. **Three-tone signal:** At $f_s=300$ Hz, write down the apparent frequencies of the components at 100, 270, and 430 Hz. Which components are above Nyquist?
5. **Prevent aliasing:** Turn on the ideal anti-alias filter. Compare the output spectrum and sound with the unfiltered case. What information has been removed? What information would otherwise be irreversibly aliased?
6. **Reconstruction:** Compare *Ideal baseband*, *Finite sinc*, *Linear*, and *Sample-and-hold*. Why do the last two differ even when $f_s>2f_{\max}$? Why does ideal reconstruction still yield the wrong analogue frequency if the sampling theorem is violated?
7. **Optional extension:** Choose *Harmonic-rich signal*. Explain why a sampling rate adequate for the fundamental may still alias higher harmonics.

**Key takeaway:** Once multiple analogue frequencies produce the same sample sequence, no post-sampling reconstruction algorithm can identify the original without further assumptions. A suitable analogue anti-alias filter must operate *before* the sampler.

**Teaching note:** The figures use continuous-time *analytical* signals and idealized line spectra. Finite sinc interpolation is an approximation; edge errors arise because only finitely many samples are summed. The ideal anti-alias filter is an illustrative component rejection operation, not a designed physical filter.